In [1]:
from ovo import db, storage, schedulers, descriptors, design_logic, descriptor_logic, \
    models_proteinqc, project_logic, descriptors_proteinqc
from ovo.app.components import molstar_notebook, StructureVisualization
import os
import seaborn as sns
from matplotlib import pyplot as plt

%config InlineBackend.figure_format = 'retina'

Registering plugin ovo_promb
Registering plugin ovo_proteindj


OVO home /home/username/ovo

In [2]:
project, project_round = project_logic.get_or_create_project_round("OVO Publication Examples 1", "Interface scaffolding")

In [3]:
pools = db.Pool.select(round_id=project_round.id)
for pool in pools:
    print(pool.id, pool.name)

xeh 5IUS 500*5 designs PD1 interface with inpaint
zuk 5IUS 500*5 designs PD1 interface


In [4]:
# Pool id from previous notebooks
POOL_IDS = [p.id for p in pools]

In [5]:
designs = db.Design.select(pool_id__in=POOL_IDS, accepted=True)
len(designs)

276

In [6]:
workflow = models_proteinqc.ProteinQCWorkflow(
    chains=['A'],
    design_ids=[design.id for design in designs],
    tools=['seq_composition', 'esm_1v', 'esm_if', 'dssp', 'peppatch', 'proteinsol'],
)
workflow.validate()
workflow.get_table_row()

Workflow  type    ProteinQC workflow
dtype: object

In [7]:
print(schedulers.keys())

SCHEDULER_KEY = 'slurm_singularity'

dict_keys(['slurm_singularity', 'pbs_singularity', 'local_singularity', 'local_conda', 'local_single_gpu'])


In [8]:
#
# SUBMIT - note that running this multiple times will submit a the workflow each time!
#
descriptor_job = descriptor_logic.submit_descriptor_workflow(
    workflow=workflow,
    scheduler_key=SCHEDULER_KEY,
    project_id=project.id
)
descriptor_job.id

Submitting workflow: nextflow run -with-trace trace.txt -work-dir /home/username/ovo/workdir/work /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc --publish_dir output --reference_files_dir /home/username/ovo/reference_files --shared_modules ovo:/home/username/projects/ovo-open-source/ovo/ovo,ovo_promb:/home/username/projects/ovo-open-source/ovo-promb/ovo_promb,ovo_proteindj:/home/username/projects/ovo-proteindj/ovo_proteindj -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/nextflow_default.config -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc/nextflow.config -profile singularity -config /home/username/ovo/nextflow_slurm_singularity.config -ansi-log false -bg --input_pdb /home/username/ovo/workdir/inputs/60/66369642ae9f3aea477b4020fb2cda8a519d7b/input_pdb_paths.txt --tools seq_composition,esm_1v,esm_if,dssp,peppatch,proteinsol --chains A --batch_size 50
Execution directory: /home/username/ovo/workdir/execdir/0ff401b0-c169

'a3df1f'

In [9]:
descriptor_logic.process_results(descriptor_job)

Waiting for job 0ff401b0-c169-11f0-b560-0248d8152725 to finish...


In [10]:
db.DescriptorValue.count(design_id__in=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True))

41952

In [11]:
values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True),
)
print(len(values), 'designs')
print(len(values.columns), 'descriptors')
values.head()

276 designs
111 descriptors


,Sequence A,AF2 ipTM score,AF2 pTM score,AF2 Design RMSD,AF2 Native Motif RMSD,AF2 PAE,AF2 pLDDT,AlphaFold2 Initial Guess prediction,Radius of gyration (backbone),Interface target residues,...,Normalized positive top1 patch area at pH 7.4,Negative patch area at pH 7.4,Normalized negative patch area at pH 7.4,Negative top1 patch area at pH 7.4,Normalized negative top1 patch area at pH 7.4,Electrostatic volume integral at pH 7.4,Positive electrostatic regions volume integral at pH 7.4,Negative electrostatic regions volume integral at pH 7.4,Positive electrostatic volume integral at pH 7.4,Negative electrostatic volume integral at pH 7.4
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_xeh_0008_seq02,AALYESRDVRVGEINLSDKISIRESPRVHTETQAETDAAVAAAMAA...,0.744985,0.794391,1.908566,1.956114,4.387716,87.074292,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,15.750123,"B18,B19,B20,B21,B22,B23,B24,B25,B26,B27,B54,B5...",...,0.026279,683.348839,0.111084,370.301777,0.060195,-10240.110952,6687.920601,-12414.339797,13125.230207,-23365.341159
ovo_xeh_0023_seq04,APAGDLRIGVIFLDDKIHIEESPRVPDDSPEARERLLAQGVAILQQ...,0.746723,0.793615,1.680266,1.971692,4.614334,84.930700,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,14.021605,"B18,B19,B20,B21,B22,B23,B24,B25,B26,B27,B54,B5...",...,0.011018,1101.154542,0.184330,365.199421,0.061133,-19816.273257,2295.776046,-13715.433234,5402.875865,-25219.149122
ovo_xeh_0025_seq05,ADAVAQVTALLERAAREEGLESLDVRMGVISLSKKISIEESPRRRV...,0.680877,0.785792,1.398022,1.945258,3.799484,85.689670,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,14.371010,"B18,B19,B20,B21,B22,B23,B24,B25,B26,B27,B54,B5...",...,0.004040,2075.170957,0.333473,1266.028591,0.203447,-39230.856128,1195.796487,-26765.925576,3002.076573,-42232.932701
ovo_xeh_0026_seq03,PVLGPREPSDFDIRAGLINLKDKISIEETPRVWADRGPRGHDAFAQ...,0.758294,0.817760,1.501997,1.952922,4.925250,85.131836,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,14.363119,"B18,B19,B20,B21,B22,B23,B24,B25,B26,B27,B54,B5...",...,0.013014,980.792725,0.155169,295.232611,0.046708,-19787.057802,2511.788356,-14040.251232,6280.290531,-26067.348333
ovo_xeh_0029_seq03,PYFPEVDPESDVRFGVIHLDDKISIEESPRIKNSDVEKINAALAAM...,0.640286,0.743111,1.583092,1.988679,4.675223,80.532986,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,12.316359,"B18,B19,B20,B21,B22,B23,B24,B25,B26,B27,B54,B5...",...,0.001755,1105.045041,0.239445,639.316527,0.138529,-24650.734607,502.436116,-14994.144542,1671.549197,-26322.283804


In [12]:
values.to_csv('../../data/results/03_rfdiffusion_pd1_interface_scaffolding/descriptors.csv')

### Save descriptors for all designs including not accepted

In [13]:
all_values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id__in=POOL_IDS),
    descriptor_keys=[d.key for d in descriptors_proteinqc.SEQ_COMPOSITION_DESCRIPTORS]
)
print(len(all_values), 'designs')
print(len(all_values.columns), 'descriptors')
all_values.head()

10000 designs
43 descriptors


,Sequence A,Sequence length,Ala %,Cys %,Asp %,Glu %,Phe %,Gly %,His %,Ile %,...,MEC reduced,MEC cystines,Helix-forming residues %,Sheet-forming residues %,Turn-forming residues %,Flexibility average,GRAVY,Instability index,Molecular weight,Sequence entropy
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_xeh_0001_seq01,APDPSLFRRVGIIHLGNKIEIEETPRQRGDADRTFSVGEGEKAHLV...,77,7.792208,0.0,5.194805,11.688312,2.597403,9.090909,5.194805,6.493506,...,0,0,31.168831,32.467532,27.272727,1.007266,-0.548052,41.016883,8410.3227,3.494483
ovo_xeh_0001_seq02,VNDPSLYRRVGVIFLGNKIEIKETPRRRGDADFTFSVKEGEKAHIV...,77,6.493506,0.0,5.194805,9.090909,3.896104,7.792208,2.597403,6.493506,...,1490,1490,29.870130,36.363636,25.974026,1.008900,-0.479221,43.944156,8536.6047,3.506382
ovo_xeh_0001_seq03,ANDPSLFRRVGVIYLGKKIEIKETPRQRGDADVTFSVGEGEKAHIV...,77,7.792208,0.0,5.194805,10.389610,2.597403,9.090909,3.896104,6.493506,...,1490,1490,32.467532,32.467532,27.272727,1.009830,-0.581818,30.085714,8437.4307,3.523938
ovo_xeh_0001_seq04,VNDPSLYRRVGVIFLGDKISIEETPRKKGDADETFKVGEGEKAHIV...,77,6.493506,0.0,7.792208,11.688312,2.597403,9.090909,2.597403,6.493506,...,1490,1490,35.064935,33.766234,28.571429,1.014926,-0.563636,25.042857,8406.3621,3.419893
ovo_xeh_0001_seq05,VNDPSLSRRVGIIHLGDKIEIEETPRQTGDADRTFSVGEGEKAHLV...,77,6.493506,0.0,6.493506,11.688312,1.298701,9.090909,5.194805,6.493506,...,0,0,28.571429,35.064935,27.272727,1.009886,-0.522078,36.419481,8331.1318,3.452458


In [14]:
all_values.to_csv('../../data/results/03_rfdiffusion_pd1_interface_scaffolding/seq_descriptors_all_designs.csv')